# News-Reel Factory — Colab voice worker (Google Drive queue)

Attended worker (blueprint §D). Mounts your Drive, drains the queue's
`pending/` folder, clones each scene's narration with **OmniVoice** on the free
T4 GPU, writes `audio/<sceneId>.wav` into the job, and moves it to `completed/`.
The Node server (gdrive backend) then renders it. Run the cells top to bottom;
leave the last one running while you author reels.

**Before running:** create the queue root folder in *My Drive* (same folder the
server points `DRIVE_ROOT_FOLDER_ID` at, e.g. `reel-queue`) and drop a clean
15–30s reference voice sample in it as `reference.wav`.

In [ ]:
# 1) Install OmniVoice (confirmed recipe: no kernel restart, no HF gate).
!pip -q install omnivoice
!pip -q install torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

In [ ]:
# 2) Mount Drive (writes happen as *you*, so free-tier storage works).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Config + load the model once.
import os, glob, subprocess, torch, torchaudio
from omnivoice import OmniVoice

QUEUE_ROOT = '/content/drive/MyDrive/reel-queue'   # must match the server's queue root folder
NUM_STEP   = 48        # 48-64 = final quality (32 = draft)
SR         = 24000     # OmniVoice output sample rate
GAP_MS     = 300       # silence inserted between sentences

# Reference voice: drop ANY clean 15-30s clip in the queue root named
# reference.<ext> (wav/mp3/m4a/flac/ogg). A .wav is used as-is; anything else is
# converted to a 24k mono wav with ffmpeg (bundled in Colab). Upload it once —
# it lives in Drive and every run reuses it, no re-upload.
cands = sorted(glob.glob(os.path.join(QUEUE_ROOT, 'reference.*')))
cands = [c for c in cands if not c.endswith('reference_prepared.wav')]
assert cands, f'No reference.* found in {QUEUE_ROOT}. Drop a 15-30s clean voice clip there.'
src = next((c for c in cands if c.lower().endswith('.wav')), cands[0])
if src.lower().endswith('.wav'):
    REF_AUDIO = src
else:
    REF_AUDIO = os.path.join(QUEUE_ROOT, 'reference_prepared.wav')
    print('converting', os.path.basename(src), '-> reference_prepared.wav')
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', src,
                    '-ac', '1', '-ar', str(SR), REF_AUDIO], check=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=torch.float16)
assert os.path.exists(REF_AUDIO), f'Reference audio not ready: {REF_AUDIO}'
print('model loaded, reference ok ->', os.path.basename(REF_AUDIO))

In [ ]:
# 4) Queue helpers + per-sentence synthesis (blueprint 'Audio quality').
import re, json, shutil, time
import numpy as np

STAGES = ['pending','in-progress','completed','generated','approved','rejected']
def stage_dir(s):
    d = os.path.join(QUEUE_ROOT, s); os.makedirs(d, exist_ok=True); return d

def split_sentences(text):
    parts = re.split(r'(?<=[.!?…।])\s+|\n+', (text or '').strip())
    return [p.strip() for p in parts if p.strip()]

def to_wav(out):
    """OmniVoice may return a torch tensor OR a numpy array depending on the
    build — normalise either to a 1-D float32 torch tensor at SR."""
    a = out[0]
    t = a.detach().float().cpu() if isinstance(a, torch.Tensor) \
        else torch.as_tensor(np.asarray(a), dtype=torch.float32)
    return t.reshape(-1)

def synth(text):
    """One generation per sentence, concatenated with fixed silence."""
    sents = split_sentences(text) or [text or ' ']
    sil = torch.zeros(int(SR * GAP_MS / 1000))
    chunks = []
    for i, sent in enumerate(sents):
        out = model.generate(text=sent, ref_audio=REF_AUDIO, num_step=NUM_STEP, speed=1.0)
        wav = to_wav(out)
        if i: chunks.append(sil)
        chunks.append(wav)
    return torch.cat(chunks) if chunks else torch.zeros(SR)

def recover_startup():
    """Single attended worker: reclaim anything stuck in-progress from a crash."""
    for jid in os.listdir(stage_dir('in-progress')):
        src = os.path.join(stage_dir('in-progress'), jid)
        if os.path.isdir(src):
            print('recovering', jid); shutil.move(src, os.path.join(stage_dir('pending'), jid))

def process(jid):
    inp = os.path.join(stage_dir('in-progress'), jid)
    shutil.move(os.path.join(stage_dir('pending'), jid), inp)   # claim by moving first
    try:
        spec = json.load(open(os.path.join(inp, 'spec.json'), encoding='utf-8'))
        adir = os.path.join(inp, 'audio'); os.makedirs(adir, exist_ok=True)
        for sc in spec['scenes']:
            wav = synth(sc.get('text', ''))
            torchaudio.save(os.path.join(adir, sc['id'] + '.wav'), wav.unsqueeze(0), SR)
            print('   ', jid, sc['id'], f'{wav.shape[-1]/SR:.1f}s')
        mp = os.path.join(inp, 'meta.json')
        try:
            m = json.load(open(mp, encoding='utf-8')); m['audioSource'] = 'final'
            json.dump(m, open(mp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
        except Exception: pass
        shutil.move(inp, os.path.join(stage_dir('completed'), jid))
        print('completed', jid)
    except Exception as e:
        print('FAILED', jid, e); shutil.move(inp, os.path.join(stage_dir('pending'), jid))

def drain_once():
    for jid in sorted(os.listdir(stage_dir('pending'))):
        if os.path.isdir(os.path.join(stage_dir('pending'), jid)):
            print('claiming', jid); process(jid)

In [ ]:
# 4b) RENDER setup (runs on Linux, so Smart App Control on the laptop is
#     irrelevant). Installs Node + Remotion + headless-chrome libs, unzips the
#     composition bundle from Drive, defines render_job(), and the heartbeat +
#     stale-recovery helpers. One-time per session.
import json, glob, shutil, subprocess, threading, wave, zipfile, datetime
import http.server, socketserver

FPS, W, H, GAP = 30, 1080, 1920, 0.35   # must match packages/composition VIDEO const
PLACEHOLDER = 3.5
BUNDLE_DIR = '/content/render-bundle'
SERVE_DIR  = '/content/serve'
SERVE_PORT = 8123
HEARTBEAT  = os.path.join(QUEUE_ROOT, 'worker-heartbeat.json')

def sh(cmd, check=True):
    print('+', cmd); return subprocess.run(cmd, shell=True, check=check)

# Heartbeat: written continuously (including *inside* a job) so the server/UI
# see the worker as Working, never falsely Offline while it's busy voicing/rendering.
def beat(status='idle'):
    try:
        json.dump(
            {'lastSeenIso': datetime.datetime.now(datetime.timezone.utc).isoformat(),
             'status': status, 'numStep': NUM_STEP},
            open(HEARTBEAT, 'w', encoding='utf-8'))
    except Exception as e:
        print('heartbeat write failed:', e)

# Node 20 (Colab may not ship it) + system libs headless Chrome needs.
if subprocess.run('node -v', shell=True).returncode != 0:
    sh('curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1')
    sh('apt-get install -y -qq nodejs >/dev/null 2>&1')
sh('apt-get update -qq >/dev/null 2>&1 && apt-get install -y -qq libnss3 libatk1.0-0 '
   'libatk-bridge2.0-0 libcups2 libdrm2 libgbm1 libasound2 libpango-1.0-0 libcairo2 '
   'libxkbcommon0 libxcomposite1 libxdamage1 libxrandr2 libxfixes3 libxi6 libgtk-3-0 '
   '>/dev/null 2>&1', check=False)

# Remotion CLI + a headless browser (downloaded once).
os.makedirs('/content/renderer', exist_ok=True)
if not os.path.isdir('/content/renderer/node_modules/@remotion'):
    open('/content/renderer/package.json', 'w').write('{"name":"r","private":true}')
    sh('cd /content/renderer && npm i --no-audit --no-fund @remotion/cli@4.0.245 @remotion/renderer@4.0.245')
sh('cd /content/renderer && npx remotion browser ensure', check=False)

# Unzip the composition bundle that the laptop built (npm run bundle).
zpath = os.path.join(QUEUE_ROOT, 'render-bundle.zip')
assert os.path.exists(zpath), f'render-bundle.zip missing in {QUEUE_ROOT} — run `npm run bundle` on the laptop.'
if os.path.isdir(BUNDLE_DIR): shutil.rmtree(BUNDLE_DIR)
os.makedirs(BUNDLE_DIR)
with zipfile.ZipFile(zpath) as z: z.extractall(BUNDLE_DIR)
print('bundle ready at', BUNDLE_DIR)

# Local static server: headless Chrome fetches media over http (file:// is blocked).
os.makedirs(SERVE_DIR, exist_ok=True)
class _H(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *a, **k): super().__init__(*a, directory=SERVE_DIR, **k)
    def log_message(self, *a): pass
threading.Thread(
    target=lambda: socketserver.TCPServer(('127.0.0.1', SERVE_PORT), _H).serve_forever(),
    daemon=True).start()
print('media server on', SERVE_PORT)

def wav_seconds(path):
    try:
        with wave.open(path) as w: return w.getnframes() / float(w.getframerate())
    except Exception: return 0.0

def build_props(inp, jid):
    """Replicate packages/composition compile(): spec + measured audio -> ReelProps."""
    spec = json.load(open(os.path.join(inp, 'spec.json'), encoding='utf-8'))
    if os.path.isdir(SERVE_DIR): shutil.rmtree(SERVE_DIR)
    a_dir = os.path.join(SERVE_DIR, 'audio'); m_dir = os.path.join(SERVE_DIR, 'media')
    os.makedirs(a_dir); os.makedirs(m_dir)
    for f in glob.glob(os.path.join(inp, 'audio', '*')): shutil.copy(f, a_dir)
    for f in glob.glob(os.path.join(inp, 'media', '*')): shutil.copy(f, m_dir)
    base = f'http://127.0.0.1:{SERVE_PORT}'
    logos = glob.glob(os.path.join(m_dir, 'logo.*'))
    logo_url = f'{base}/media/{os.path.basename(logos[0])}' if logos else ''
    scenes, cursor = [], 0
    for s in spec['scenes']:
        wavp = os.path.join(a_dir, s['id'] + '.wav')
        secs = wav_seconds(wavp) if os.path.exists(wavp) else PLACEHOLDER
        dif = max(1, round((secs + GAP) * FPS))
        imgs = glob.glob(os.path.join(m_dir, s['id'] + '.*'))
        scenes.append({
            'id': s['id'],
            'image': f"{base}/media/{os.path.basename(imgs[0])}" if imgs else '',
            'caption': s.get('caption', ''),
            'stat': s.get('stat'),
            'audio': f'{base}/audio/{s["id"]}.wav' if os.path.exists(wavp) else '',
            'from': cursor, 'durationInFrames': dif,
        })
        cursor += dif
    b = spec.get('brand', {})
    props = {'id': spec['id'], 'fps': FPS, 'width': W, 'height': H, 'totalFrames': cursor,
             'brand': {'kicker': b.get('kicker', ''), 'date': b.get('date', ''),
                       'name': b.get('name', ''), 'logo': logo_url},
             'scenes': scenes}
    pp = os.path.join(inp, 'props.json')
    json.dump(props, open(pp, 'w', encoding='utf-8'), ensure_ascii=False)
    return pp

def render_job(inp, jid):
    pp = build_props(inp, jid)
    out = os.path.join(inp, jid + '.mp4')
    sh(f'cd /content/renderer && npx remotion render "{BUNDLE_DIR}" Reel "{out}" '
       f'--props="{pp}" --concurrency=2 --log=error')
    print('rendered', jid, '->', out)
    return out

def recover_stale(max_age_sec=2400):
    """Reclaim jobs stuck in-progress (a previous session crashed mid-job) older
    than ~40 min, back to pending. Safe here: single worker, so an in-progress
    job seen while the loop is idle is an orphan, not a live one."""
    d = stage_dir('in-progress')
    for jid in os.listdir(d):
        p = os.path.join(d, jid)
        if os.path.isdir(p) and (time.time() - os.path.getmtime(p)) > max_age_sec:
            print('recovering stale', jid); shutil.move(p, os.path.join(stage_dir('pending'), jid))

# Redefine process() to voice AND render, heartbeat continuously, and dead-letter
# a poison job after 3 attempts (so it can't loop forever burning GPU).
def process(jid):
    inp = os.path.join(stage_dir('in-progress'), jid)
    shutil.move(os.path.join(stage_dir('pending'), jid), inp)  # claim by moving
    try:
        beat('voicing ' + jid)
        spec = json.load(open(os.path.join(inp, 'spec.json'), encoding='utf-8'))
        adir = os.path.join(inp, 'audio'); os.makedirs(adir, exist_ok=True)
        for sc in spec['scenes']:
            wav = synth(sc.get('text', ''))
            torchaudio.save(os.path.join(adir, sc['id'] + '.wav'), wav.unsqueeze(0), SR)
            print('   voiced', jid, sc['id'], f'{wav.shape[-1]/SR:.1f}s')
            beat('voicing ' + jid)          # refresh heartbeat between scenes
        mp = os.path.join(inp, 'meta.json')
        try:
            m = json.load(open(mp, encoding='utf-8')); m['audioSource'] = 'final'
            json.dump(m, open(mp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
        except Exception: pass
        beat('rendering ' + jid)            # heartbeat before the long render
        render_job(inp, jid)
        shutil.move(inp, os.path.join(stage_dir('completed'), jid))
        print('completed', jid)
    except Exception as e:
        print('FAILED', jid, e)
        # dead-letter: count attempts; after 3, send to rejected/ with the reason
        # (surfaces in the review UI) instead of looping pending->in-progress forever.
        try:
            m = json.load(open(os.path.join(inp, 'meta.json'), encoding='utf-8'))
        except Exception:
            m = {}
        attempts = int(m.get('attempts', 0)) + 1
        m['attempts'] = attempts
        m['lastError'] = str(e)[:500]
        to = 'rejected' if attempts >= 3 else 'pending'
        if to == 'rejected':
            m['rejectReason'] = f'failed {attempts}x: {str(e)[:300]}'
        try:
            json.dump(m, open(os.path.join(inp, 'meta.json'), 'w', encoding='utf-8'),
                      ensure_ascii=False, indent=2)
        except Exception: pass
        shutil.move(inp, os.path.join(stage_dir(to), jid))
        print(f'   -> {to} (attempt {attempts})')

print('render setup done.')

In [ ]:
# 5) Run the worker loop. Ctrl+C / stop the cell to end. Heartbeats are written
#    continuously — even mid-job (per scene + before render) — so the web UI shows
#    "Working", never a false "Offline". recover_stale reclaims jobs orphaned by a
#    disconnect. (beat / recover_stale / recover_startup are defined above.)
recover_startup()          # on boot, reclaim anything left in-progress by a hard crash
beat('starting')
print('draining queue every 10s (heartbeat on)...')
while True:
    try:
        recover_stale()    # reclaim mid-session orphans (>~40 min in-progress)
        beat('draining')
        drain_once()
        beat('idle')
    except Exception as e:
        print('loop error:', e)
    time.sleep(10)